In [4]:
# Imports
import numpy as np
import nashpy as nash
from scipy.optimize import linprog
import matplotlib.pyplot as plt
from typing import Tuple, Optional
import warnings
warnings.filterwarnings('ignore')
print("Imports OK : numpy, scipy.optimize")

Imports OK : numpy, scipy.optimize


## 1. Initialisation ZeroSumGame

In [8]:
class ZeroSumGame:
    """
    Jeu a somme nulle a deux joueurs.
    
    La matrice A represente les gains de Row.
    Les gains de Col sont -A.
    """
    
    def __init__(self, A: np.ndarray, 
                 row_labels: list = None,
                 col_labels: list = None,
                 name: str = "Zero-Sum Game"):
        self.A = np.array(A, dtype=float)
        self.m, self.n = self.A.shape
        self.name = name
        self.row_labels = row_labels or [f"R{i}" for i in range(self.m)]
        self.col_labels = col_labels or [f"C{j}" for j in range(self.n)]
    
    def payoff(self, sigma_row: np.ndarray, sigma_col: np.ndarray) -> float:
        """Gain de Row (= -gain de Col)."""
        return sigma_row @ self.A @ sigma_col
    
    def display(self):
        """Affiche la matrice des gains."""
        print(f"\n{self.name} (gains de Row)")
        print("=" * 50)
        
        # Header
        header = "        " + "  ".join(f"{c:>8}" for c in self.col_labels)
        print(header)
        print("-" * len(header))
        
        # Rows
        for i, label in enumerate(self.row_labels):
            row_str = f"{label:>6}  "
            row_str += "  ".join(f"{self.A[i,j]:>8.2f}" for j in range(self.n))
            print(row_str)

# Exemples de jeux a somme nulle

# Pierre-Feuille-Ciseaux
rps = ZeroSumGame(
    A=[[0, -1, 1],
       [1, 0, -1],
       [-1, 1, 0]],
    row_labels=['Pierre', 'Feuille', 'Ciseaux'],
    col_labels=['Pierre', 'Feuille', 'Ciseaux'],
    name="Pierre-Feuille-Ciseaux"
)
rps.display()

# Matching Pennies
mp = ZeroSumGame(
    A=[[1, -1],
       [-1, 1]],
    row_labels=['Pile', 'Face'],
    col_labels=['Pile', 'Face'],
    name="Matching Pennies"
)
mp.display()


Pierre-Feuille-Ciseaux (gains de Row)
          Pierre   Feuille   Ciseaux
------------------------------------
Pierre      0.00     -1.00      1.00
Feuille      1.00      0.00     -1.00
Ciseaux     -1.00      1.00      0.00

Matching Pennies (gains de Row)
            Pile      Face
--------------------------
  Pile      1.00     -1.00
  Face     -1.00      1.00


## 2. Theoreme Minimax de Von Neumann

### Enonce

Pour tout jeu matriciel a somme nulle, en **strategies mixtes** :

$$\max_{\sigma \in \Delta_m} \min_{\tau \in \Delta_n} \sigma^T A \tau = \min_{\tau \in \Delta_n} \max_{\sigma \in \Delta_m} \sigma^T A \tau = v$$

ou $\Delta_k$ est le simplexe des probabilites en dimension $k$.

### Consequences

1. La **valeur du jeu** $v$ est bien definie
2. Il existe des strategies **optimales** $\sigma^*$ et $\tau^*$
3. Ces strategies forment un **equilibre de Nash**

In [9]:
def solve_minimax_lp(game: ZeroSumGame) -> Tuple[np.ndarray, float]:
    """
    Resout le probleme minimax par programmation lineaire.
    
    Probleme primal (Row maximise):
        max v
        s.t. A @ sigma >= v * 1
             sigma >= 0, sum(sigma) = 1
    
    Reformule comme:
        min -v
        s.t. -A.T @ sigma + v * 1 <= 0
             sum(sigma) = 1
             sigma >= 0
    
    Returns:
        (strategie optimale de Row, valeur du jeu)
    """
    m, n = game.A.shape
    
    # Variables: [sigma_0, ..., sigma_{m-1}, v]
    # Objectif: max v <=> min -v
    c = np.zeros(m + 1)
    c[-1] = -1  # min -v
    
    # Contraintes d'inegalite: A.T @ sigma >= v * 1
    # Reformule: -A.T @ sigma + v <= 0
    A_ub = np.zeros((n, m + 1))
    A_ub[:, :m] = -game.A.T
    A_ub[:, m] = 1
    b_ub = np.zeros(n)
    
    # Contrainte d'egalite: sum(sigma) = 1
    A_eq = np.zeros((1, m + 1))
    A_eq[0, :m] = 1
    b_eq = np.array([1])
    
    # Bornes: sigma >= 0, v non borne
    bounds = [(0, None) for _ in range(m)] + [(None, None)]
    
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    
    if result.success:
        sigma = result.x[:m]
        v = result.x[m]
        return sigma, v
    else:
        raise ValueError(f"LP failed: {result.message}")

def solve_minimax_dual(game: ZeroSumGame) -> Tuple[np.ndarray, float]:
    """
    Resout le probleme dual (Col minimise).
    
    Returns:
        (strategie optimale de Col, valeur du jeu)
    """
    # Le dual pour Col sur -A.T est equivalent au primal pour Row sur A.T
    m, n = game.A.shape
    
    # Variables: [tau_0, ..., tau_{n-1}, w]
    # Objectif: min w
    c = np.zeros(n + 1)
    c[-1] = 1  # min w
    
    # Contraintes: A @ tau <= w * 1
    A_ub = np.zeros((m, n + 1))
    A_ub[:, :n] = game.A
    A_ub[:, n] = -1
    b_ub = np.zeros(m)
    
    # Contrainte d'egalite: sum(tau) = 1
    A_eq = np.zeros((1, n + 1))
    A_eq[0, :n] = 1
    b_eq = np.array([1])
    
    # Bornes
    bounds = [(0, None) for _ in range(n)] + [(None, None)]
    
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    
    if result.success:
        tau = result.x[:n]
        w = result.x[n]
        return tau, w
    else:
        raise ValueError(f"LP failed: {result.message}")

# Resoudre Matching Pennies
print("Resolution de Matching Pennies")
print("=" * 50)
mp.display()

sigma, v_primal = solve_minimax_lp(mp)
tau, v_dual = solve_minimax_dual(mp)

print(f"\nSolution:")
print(f"  Strategie Row: {np.round(sigma, 4)}")
print(f"  Valeur (primal): {v_primal:.4f}")
print(f"  Strategie Col: {np.round(tau, 4)}")
print(f"  Valeur (dual): {v_dual:.4f}")
print(f"\nVerification theoreme minimax: primal = dual ? {abs(v_primal - v_dual) < 1e-6}")

Resolution de Matching Pennies

Matching Pennies (gains de Row)
            Pile      Face
--------------------------
  Pile      1.00     -1.00
  Face     -1.00      1.00

Solution:
  Strategie Row: [0.5 0.5]
  Valeur (primal): -0.0000
  Strategie Col: [0.5 0.5]
  Valeur (dual): -0.0000

Verification theoreme minimax: primal = dual ? True


### Interpretation : Resolution par programmation lineaire

La resolution de Matching Pennies par LP confirme le theoreme minimax :

**Resultats** :
| Element | Valeur | Signification |
|---------|--------|---------------|
| Strategie Row | `[0.5, 0.5]` | Jouer Pile et Face avec probabilite egale |
| Strategie Col | `[0.5, 0.5]` | Meme strategie optimale (jeu symetrique) |
| Valeur du jeu | 0 | Ni avantage ni desavantage pour Row |
| Primal = Dual | Oui | Verification du theoreme de dualite forte |

**Pourquoi 50/50 ?** Si Row jouait Pile plus souvent, Col pourrait exploiter ce biais en jouant Face systematiquement. L'equilibre est atteint quand aucun joueur ne peut ameliorer sa situation en changeant sa strategie.

> **Note technique** : La formulation LP transforme le probleme "max-min" en un probleme d'optimisation standard, resoluble en temps polynomial par l'algorithme du simplexe.

In [10]:
# Resoudre Pierre-Feuille-Ciseaux
print("Resolution de Pierre-Feuille-Ciseaux")
print("=" * 50)
rps.display()

sigma, v_primal = solve_minimax_lp(rps)
tau, v_dual = solve_minimax_dual(rps)

print(f"\nSolution:")
print(f"  Strategie Row: {np.round(sigma, 4)}")
print(f"  Valeur du jeu: {v_primal:.4f}")
print(f"  Strategie Col: {np.round(tau, 4)}")

# Verification par Nashpy
rps_nash = nash.Game(rps.A, -rps.A)
for eq in rps_nash.support_enumeration():
    print(f"\nVerification Nashpy: {np.round(eq[0], 4)}, {np.round(eq[1], 4)}")

Resolution de Pierre-Feuille-Ciseaux

Pierre-Feuille-Ciseaux (gains de Row)
          Pierre   Feuille   Ciseaux
------------------------------------
Pierre      0.00     -1.00      1.00
Feuille      1.00      0.00     -1.00
Ciseaux     -1.00      1.00      0.00

Solution:
  Strategie Row: [0.3333 0.3333 0.3333]
  Valeur du jeu: -0.0000
  Strategie Col: [0.3333 0.3333 0.3333]

Verification Nashpy: [0.3333 0.3333 0.3333], [0.3333 0.3333 0.3333]


## 3. Dualite en programmation lineaire

Le theoreme minimax est intimement lie a la **dualite LP**.

### Probleme primal (Row maximise)

$$\begin{aligned}
\max & \quad v \\
\text{s.t.} & \quad \sum_i \sigma_i A_{ij} \geq v \quad \forall j \\
& \quad \sum_i \sigma_i = 1 \\
& \quad \sigma_i \geq 0
\end{aligned}$$

### Probleme dual (Col minimise)

$$\begin{aligned}
\min & \quad w \\
\text{s.t.} & \quad \sum_j A_{ij} \tau_j \leq w \quad \forall i \\
& \quad \sum_j \tau_j = 1 \\
& \quad \tau_j \geq 0
\end{aligned}$$

Par dualite forte : $v^* = w^*$.

In [11]:
def verify_duality(game: ZeroSumGame):
    """Verifie la dualite LP pour le theoreme minimax."""
    print(f"\nVerification de la dualite pour: {game.name}")
    print("=" * 50)
    
    sigma_opt, v_primal = solve_minimax_lp(game)
    tau_opt, v_dual = solve_minimax_dual(game)
    
    print(f"\nProbleme primal (Row maximise):")
    print(f"  sigma* = {np.round(sigma_opt, 4)}")
    print(f"  v* = {v_primal:.6f}")
    
    print(f"\nProbleme dual (Col minimise):")
    print(f"  tau* = {np.round(tau_opt, 4)}")
    print(f"  w* = {v_dual:.6f}")
    
    print(f"\nDualite forte: v* = w* ? {abs(v_primal - v_dual) < 1e-6}")
    
    # Verification complementary slackness
    print(f"\nVerification des contraintes:")
    print(f"  Gains de Row pour chaque action Col:")
    gains_row = sigma_opt @ game.A
    for j, g in enumerate(gains_row):
        print(f"    Col joue {game.col_labels[j]}: {g:.4f} >= {v_primal:.4f} ? {g >= v_primal - 1e-6}")

verify_duality(rps)
verify_duality(mp)


Verification de la dualite pour: Pierre-Feuille-Ciseaux

Probleme primal (Row maximise):
  sigma* = [0.3333 0.3333 0.3333]
  v* = -0.000000

Probleme dual (Col minimise):
  tau* = [0.3333 0.3333 0.3333]
  w* = -0.000000

Dualite forte: v* = w* ? True

Verification des contraintes:
  Gains de Row pour chaque action Col:
    Col joue Pierre: 0.0000 >= -0.0000 ? True
    Col joue Feuille: 0.0000 >= -0.0000 ? True
    Col joue Ciseaux: -0.0000 >= -0.0000 ? True

Verification de la dualite pour: Matching Pennies

Probleme primal (Row maximise):
  sigma* = [0.5 0.5]
  v* = -0.000000

Probleme dual (Col minimise):
  tau* = [0.5 0.5]
  w* = -0.000000

Dualite forte: v* = w* ? True

Verification des contraintes:
  Gains de Row pour chaque action Col:
    Col joue Pile: 0.0000 >= -0.0000 ? True
    Col joue Face: 0.0000 >= -0.0000 ? True


In [13]:
print("="*60)
print("Comparaison LP vs NashPy")
print("="*60)

sigma_lp, v_lp = solve_minimax_lp(rps)
tau_lp, _ = solve_minimax_dual(rps)

game = nash.Game(rps.A, -rps.A)
sigma_nash, tau_nash = next(game.support_enumeration())

print("Programmation linéaire")
print("Row :", np.round(sigma_lp,4))
print("Col :", np.round(tau_lp,4))
print("Valeur :", round(v_lp,4))

print()

print("NashPy")
print("Row :", np.round(sigma_nash,4))
print("Col :", np.round(tau_nash,4))

print()

print("Même stratégie Row :", np.allclose(sigma_lp,sigma_nash))
print("Même stratégie Col :", np.allclose(tau_lp,tau_nash))

Comparaison LP vs NashPy
Programmation linéaire
Row : [0.3333 0.3333 0.3333]
Col : [0.3333 0.3333 0.3333]
Valeur : -0.0

NashPy
Row : [0.3333 0.3333 0.3333]
Col : [0.3333 0.3333 0.3333]

Même stratégie Row : True
Même stratégie Col : True


# Comparaison entre les deux approches

Dans un **jeu à somme nulle**, les intérêts des deux joueurs sont strictement opposés :

- **Row** cherche à **maximiser son gain minimal** (principe du minimax).
- **Col** cherche à **minimiser le gain maximal** de Row.

Le **théorème minimax de Von Neumann** établit que ces deux objectifs sont équivalents :

$$
\max_{\sigma}\min_{\tau}\sigma^{T}A\tau
=
\min_{\tau}\max_{\sigma}\sigma^{T}A\tau.
$$

Cette égalité est garantie par la **dualité de la programmation linéaire** : le problème primal (joueur Row) et son problème dual (joueur Col) possèdent la même valeur optimale.

Un **équilibre de Nash** est précisément le couple de stratégies mixtes optimales

$$
(\sigma^*,\tau^*)
$$

obtenu par ces deux programmes linéaires.

## Comparaison des méthodes

| Programmation linéaire | Formulation directe de Nash |
| :--- | :--- |
| Résout un problème d'optimisation (primal/dual). | Recherche directement les équilibres de Nash (avec **NashPy**). |
| Fournit les stratégies optimales et la valeur du jeu. | Fournit les stratégies d'équilibre. |
| Repose sur le théorème minimax et la dualité. | Repose sur la définition d'un équilibre de Nash. |

## Conclusion

Les expériences réalisées montrent que :

- la **programmation linéaire** calcule les stratégies mixtes optimales ;
- **NashPy** retrouve exactement le même équilibre de Nash ;
- les probabilités obtenues pour les deux joueurs sont identiques ;
- la valeur du jeu est la même dans les deux approches.

Ces résultats illustrent concrètement le **théorème minimax de Von Neumann** : dans un jeu à somme nulle, **résoudre le problème minimax par programmation linéaire est équivalent à calculer un équilibre de Nash**.